# MPC (MPPI) Baseline

Trains a learned dynamics model and evaluates an MPPI controller on the drone
environment, for comparison against MAPPO.

Uses `UnityMultiAgentEnv` (low-level ML-Agents adapter), so each drone's
episode ends and respawns independently — same as MAPPO training.

**State frame note:** the env observes NO world positions. The drone-state
sensor is `[localGoalPos(3), rotationQuat(4), localVel(3), localAngVel(3), speed(1)]`,
so the MPC plans in the drone's local frame and the goal is always the
ORIGIN of that frame (localGoalPos -> 0).

In [15]:
import numpy as np
import torch
import wandb
from pathlib import Path

from mlagents_envs.side_channel.stats_side_channel import StatsSideChannel

from MAPPO.unity_env import UnityMultiAgentEnv
from MPC.MPC_Agent import MultiAgentMPC, MPCConfig

In [16]:
class Args:
    def __init__(self, d):
        self.__dict__ = d

args = Args({
    'env_path': 'Env/FinalLevel/DroneFlightv1',
    'exploration_steps': 5000,      # random actions to seed the dynamics buffer
    'max_steps': 50000,             # total env steps (exploration + MPC)
    'log_every': 1000,
    'eval_episodes': 100,
    'success_reward_threshold': 15.0,  # same proxy as MAPPO/train.py
    'save_dir': 'saved_models_mpc',
    'seed': 42,
    'no_graphics': True,
    'use_wandb': False,
})

np.random.seed(args.seed)
torch.manual_seed(args.seed)

In [17]:
print("Loading Unity environment...")
stats_channel = StatsSideChannel()
env = UnityMultiAgentEnv(
    file_name=args.env_path, seed=args.seed,
    no_graphics=args.no_graphics, side_channels=[stats_channel])

num_agents = env.num_agents
print(f"  Agents: {num_agents} | Camera: {env.camera_shape} | "
      f"Vector dim: {env.vector_dim} | Action dim: {env.action_dim}")

# Locate the drone-state block inside the concatenated vector obs.
# Layout (verified with MPC/calibrate.py): [ raycast(36) | drone-state ].
# The drone-state sensor is 14-dim AND frame-stacked (x4 -> 56 dims), so it
# is LARGER than the raycast block — the old "smallest block" heuristic wrongly
# picked the raycast. The CURRENT frame is the last FRAME_SIZE entries, and its
# first 3 values are localGoalPos/50 (x=right, y=up, z=forward).
FRAME_SIZE = 14
kin_start = env.vector_dim - FRAME_SIZE   # start of the most recent frame
kin_dim = FRAME_SIZE
print(f"  Vector blocks: {env.vector_block_slices}")
print(f"  Drone-state current frame: offset={kin_start}, dim={kin_dim} "
      f"(localGoalPos at [{kin_start}:{kin_start+3}])")

config = MPCConfig(
    horizon=20,
    dynamics_lr=1e-3,
    action_dim=env.action_dim,
    kinematic_dim=kin_dim,
    kinematic_offset=kin_start,
)

mpc_agent = MultiAgentMPC(num_agents=num_agents, config=config)

# The goal in the drone's local frame is always the origin (state[:3] is the
# local goal offset, which the controller drives to zero).
GOALS = np.zeros((num_agents, 3), dtype=np.float32)

if args.use_wandb:
    wandb.init(project="MAPPO_Drones", config={'baseline': 'MPC', **args.__dict__})

print("✓ MPC initialized")

Loading Unity environment...
  Agents: 4 | Camera: (4, 84, 84) | Vector dim: 92 | Action dim: 4
  Vector blocks: [(0, 36), (36, 92)]
  Drone-state current frame: offset=78, dim=14 (localGoalPos at [78:81])
✓ MPC initialized


In [18]:
def train_dynamics_model(mpc_agent, env, args):
    """Phase 1: random exploration seeds the dynamics buffer.
       Phase 2: MPPI control with continual dynamics learning."""
    episode_rewards, episode_lengths, episode_successes = [], [], []
    solve_times, dynamics_losses = [], []

    agent_episode_reward = np.zeros(env.num_agents)
    agent_episode_length = np.zeros(env.num_agents, dtype=np.int64)

    cam, vec = env.reset()
    total_steps = 0

    while total_steps < args.max_steps:
        exploring = total_steps < args.exploration_steps

        if exploring:
            actions = np.random.uniform(
                -1, 1, (env.num_agents, env.action_dim)).astype(np.float32)
        else:
            actions, info = mpc_agent.get_actions(vec, GOALS)
            actions = actions.astype(np.float32)
            solve_times.append(info['solve_time'])

        cam, vec_next, rewards, dones, interrupted, stale = env.step(actions)

        # Train the dynamics model on real transitions only: a respawn jump
        # (terminal step) is not dynamics, so done agents are excluded.
        transition = {i: {'state': vec[i], 'action': actions[i], 'next_state': vec_next[i]}
                      for i in range(env.num_agents) if dones[i] < 0.5 and stale[i] < 0.5}
        if transition:
            loss = mpc_agent.update_dynamics([transition])
            if loss > 0:
                dynamics_losses.append(float(loss))

        agent_episode_reward += rewards
        agent_episode_length += 1
        for i in np.flatnonzero(dones > 0.5):
            episode_rewards.append(float(agent_episode_reward[i]))
            episode_lengths.append(int(agent_episode_length[i]))
            episode_successes.append(float(rewards[i] > args.success_reward_threshold))
            agent_episode_reward[i] = 0.0
            agent_episode_length[i] = 0

        vec = vec_next
        total_steps += 1

        if total_steps % args.log_every == 0:
            mean_r = np.mean(episode_rewards[-100:]) if episode_rewards else 0.0
            mean_l = np.mean(episode_lengths[-100:]) if episode_lengths else 0.0
            succ = np.mean(episode_successes[-100:]) if episode_successes else 0.0
            phase = "explore" if exploring else "MPC"
            print(f"Step {total_steps:,} [{phase}] | reward {mean_r:8.2f} | "
                  f"success {succ:6.1%} | ep len {mean_l:6.1f}"
                  + (f" | dyn loss {dynamics_losses[-1]:.4f}" if dynamics_losses else "")
                  + (f" | solve {np.mean(solve_times[-100:])*1000:.1f} ms" if solve_times else ""))
            if args.use_wandb:
                wandb.log({
                    'mpc/reward_mean': mean_r,
                    'mpc/success_rate': succ,
                    'mpc/episode_length': mean_l,
                    'mpc/dynamics_loss': dynamics_losses[-1] if dynamics_losses else 0.0,
                    'mpc/solve_time_ms': np.mean(solve_times[-100:]) * 1000 if solve_times else 0.0,
                }, step=total_steps)

    return episode_rewards, solve_times, dynamics_losses

In [19]:
def evaluate_mpc(mpc_agent, env, args):
    """Evaluate the trained MPPI controller; success/reward per agent-episode,
       same definitions as MAPPO/train.py for a fair comparison."""
    episode_rewards, episode_lengths, successes = [], [], []
    solve_times = []

    agent_episode_reward = np.zeros(env.num_agents)
    agent_episode_length = np.zeros(env.num_agents, dtype=np.int64)

    cam, vec = env.reset()
    while len(episode_rewards) < args.eval_episodes:
        actions, info = mpc_agent.get_actions(vec, GOALS)
        solve_times.append(info['solve_time'])
        cam, vec, rewards, dones, interrupted, stale = env.step(actions.astype(np.float32))

        agent_episode_reward += rewards
        agent_episode_length += 1
        for i in np.flatnonzero(dones > 0.5):
            episode_rewards.append(float(agent_episode_reward[i]))
            episode_lengths.append(int(agent_episode_length[i]))
            successes.append(float(rewards[i] > args.success_reward_threshold))
            agent_episode_reward[i] = 0.0
            agent_episode_length[i] = 0

    print("=" * 60)
    print("MPC Evaluation Results")
    print("=" * 60)
    print(f"Episodes:       {len(episode_rewards)}")
    print(f"Mean reward:    {np.mean(episode_rewards):.2f} ± {np.std(episode_rewards):.2f}")
    print(f"Success rate:   {np.mean(successes):.1%}")
    print(f"Mean length:    {np.mean(episode_lengths):.1f}")
    print(f"Avg solve time: {np.mean(solve_times) * 1000:.1f} ms")
    return {'rewards': episode_rewards, 'success_rate': float(np.mean(successes)),
            'solve_times': solve_times}

In [20]:
episode_rewards, solve_times, dynamics_losses = train_dynamics_model(mpc_agent, env, args)

save_dir = Path(args.save_dir)
save_dir.mkdir(parents=True, exist_ok=True)
mpc_agent.save(save_dir / "mpc_final.pth")
print(f"✓ MPC dynamics models saved to {save_dir / 'mpc_final.pth'}")

results = evaluate_mpc(mpc_agent, env, args)

env.close()
if args.use_wandb:
    wandb.finish()

Step 1,000 [explore] | reward    -3.14 | success   0.0% | ep len   98.2 | dyn loss 0.0000
Step 2,000 [explore] | reward    -4.29 | success   0.0% | ep len   95.2 | dyn loss 0.0000
Step 3,000 [explore] | reward    -3.42 | success   0.0% | ep len   99.6 | dyn loss 0.0000
Step 4,000 [explore] | reward    -4.08 | success   0.0% | ep len  100.5 | dyn loss 0.0000
Step 5,000 [explore] | reward    -4.73 | success   0.0% | ep len   98.2 | dyn loss 0.0000
Step 6,000 [MPC] | reward    23.17 | success  71.0% | ep len   27.7 | dyn loss 0.0000 | solve 13.6 ms
Step 7,000 [MPC] | reward    24.45 | success  79.0% | ep len   32.1 | dyn loss 0.0002 | solve 13.8 ms
Step 8,000 [MPC] | reward    23.48 | success  71.0% | ep len   28.2 | dyn loss 0.0000 | solve 13.6 ms
Step 9,000 [MPC] | reward    25.79 | success  81.0% | ep len   30.4 | dyn loss 0.0000 | solve 13.7 ms
Step 10,000 [MPC] | reward    25.83 | success  82.0% | ep len   30.3 | dyn loss 0.0000 | solve 13.5 ms
Step 11,000 [MPC] | reward    24.95 | s

## Privileged proportional-controller baseline

The learned-dynamics MPPI above fails (below random — accurate forward dynamics
can't be learned in the drone's rotating egocentric frame), which is reported
as a negative result. As a *working* classical reference we use a **privileged
proportional controller**: it reads the goal direction directly from the
observation (`localGoalPos`) and steers straight at it. It has no obstacle
sensing, so it should solve open levels and fail on cluttered ones.

It needs no dynamics training — instantiate and evaluate directly. Change
`args.env_path` and re-run the env-setup cell to evaluate per level.

In [21]:
from MPC.MPC_Agent import MultiAgentProportionalController, MPCConfig

# The MPPI cell closes `env`, so (re)open a fresh one (closing any existing
# one first to avoid a worker-port clash).
try:
    env.close()
except Exception:
    pass

env = UnityMultiAgentEnv(
    file_name=args.env_path, seed=args.seed,
    no_graphics=args.no_graphics, side_channels=[StatsSideChannel()])
num_agents = env.num_agents

# Rebuild config HERE so the correct kinematic offset is always used, even if
# the env-setup cell above was not re-run. The drone-state sensor is stacked,
# so the current frame is the last 14 entries; localGoalPos is its first 3
# (verified with MPC/calibrate.py).
FRAME_SIZE = 14
config = MPCConfig(
    horizon=20, dynamics_lr=1e-3, action_dim=env.action_dim,
    kinematic_dim=FRAME_SIZE, kinematic_offset=env.vector_dim - FRAME_SIZE)
print(f"localGoalPos read from state[{config.kinematic_offset}:"
      f"{config.kinematic_offset + 3}]  (must NOT be 0)")

GOALS = np.zeros((num_agents, 3), dtype=np.float32)
p_agent = MultiAgentProportionalController(num_agents=num_agents, config=config)
p_results = evaluate_mpc(p_agent, env, args)

env.close()

localGoalPos read from state[78:81]  (must NOT be 0)
MPC Evaluation Results
Episodes:       100
Mean reward:    22.19 ± 28.53
Success rate:   67.0%
Mean length:    24.2
Avg solve time: 0.1 ms
